# CRSP Universe Validation

This notebook validates the market-side candidate universe built from WRDS/CRSP.

It focuses on the current milestone: confirming that the top-volume candidate list is large enough and sensible before moving on to the full daily OHLCV pull or TRNA news filtering.

The universe file is produced by:

```bash
conda run -n sentiment-ltr-paper python scripts/build_crsp_market_universe.py
```

In [1]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

UNIVERSE_PATH = PROJECT_ROOT / "data" / "raw" / "market" / "crsp_top_volume_universe.csv"
MANIFEST_PATH = PROJECT_ROOT / "data" / "raw" / "market" / "crsp_top_volume_universe_manifest.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Universe file exists: {UNIVERSE_PATH.exists()}")
print(f"Manifest file exists: {MANIFEST_PATH.exists()}")

Project root: /Users/armandoordoricadelatorre/Documents/U of T/PhD/PhD Research/Sentiment_learn_to_rank_paper
Universe file exists: True
Manifest file exists: True


In [2]:
if not UNIVERSE_PATH.exists():
    raise FileNotFoundError(
        "CRSP universe file is missing. Run scripts/build_crsp_market_universe.py first."
    )

universe = pd.read_csv(
    UNIVERSE_PATH,
    parse_dates=["first_trade_date", "last_trade_date", "latest_name_start", "latest_name_end"],
)

with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

print(f"Rows: {len(universe):,}")
print(f"Expected rows from manifest: {manifest['candidate_count']:,}")
print(f"Date range: {manifest['start']} to {manifest['end']}")
print(f"Share codes: {manifest['share_codes']}")
print(f"Exchange codes: {manifest['exchange_codes']}")

universe.head()

Rows: 1,000
Expected rows from manifest: 1,000
Date range: 2003-01-01 to 2014-12-31
Share codes: [10, 11]
Exchange codes: [1, 2, 3]


,volume_rank,permno,permco,ticker,comnam,shrcd,exchcd,trading_days,first_trade_date,last_trade_date,avg_volume,avg_dollar_volume,avg_abs_price,avg_shares_outstanding,latest_name_start,latest_name_end
0,1,70519,20483,C,CITIGROUP INC,11,1,3017,2003-01-02,2014-12-31,1.296605e+08,1.341212e+09,34.406895,7.525640e+06,2014-07-31,2015-02-04
1,2,59408,3151,BAC,BANK OF AMERICA CORP,11,1,3021,2003-01-02,2014-12-31,1.062559e+08,1.530118e+09,32.937791,6.753939e+06,2014-05-14,2015-06-30
2,3,10107,8048,MSFT,MICROSOFT CORP,11,3,3021,2003-01-02,2014-12-31,6.085280e+07,1.698544e+09,28.871619,9.287396e+06,2004-06-10,2024-03-11
3,4,59328,2367,INTC,INTEL CORP,11,3,3021,2003-01-02,2014-12-31,5.830431e+07,1.301339e+09,22.974088,5.640616e+06,2004-06-10,2024-12-31
4,5,80924,13304,SIRI,SIRIUS X M HOLDINGS INC,11,3,3021,2003-01-02,2014-12-31,5.490277e+07,1.499572e+08,2.795693,3.002844e+06,2013-11-15,2023-04-10


## Basic Validation Checks

These checks verify that the candidate universe is large enough, ordered correctly, and restricted to the intended CRSP common-share and exchange filters.

In [3]:
expected_share_codes = set(manifest["share_codes"])
expected_exchange_codes = set(manifest["exchange_codes"])

checks = {
    "has_expected_row_count": len(universe) == manifest["candidate_count"],
    "volume_rank_is_unique": universe["volume_rank"].is_unique,
    "permno_is_unique": universe["permno"].is_unique,
    "avg_volume_descending": universe["avg_volume"].is_monotonic_decreasing,
    "share_codes_are_expected": set(universe["shrcd"].dropna().astype(int)).issubset(expected_share_codes),
    "exchange_codes_are_expected": set(universe["exchcd"].dropna().astype(int)).issubset(expected_exchange_codes),
}

validation = pd.Series(checks, name="passed").to_frame()
validation

,passed
has_expected_row_count,True
volume_rank_is_unique,True
permno_is_unique,True
avg_volume_descending,True
share_codes_are_expected,False
exchange_codes_are_expected,False


## Top 20 Stocks By Average Trading Volume

The table and chart below show the highest average daily CRSP share volume over 2003-2014. This should be interpreted as the market-side candidate ranking only. It has not yet been filtered by news coverage.

In [4]:
top20 = universe.head(20).copy()
top20["avg_volume_millions"] = top20["avg_volume"] / 1_000_000
top20["avg_dollar_volume_billions"] = top20["avg_dollar_volume"] / 1_000_000_000

columns = [
    "volume_rank",
    "permno",
    "ticker",
    "comnam",
    "trading_days",
    "avg_volume_millions",
    "avg_dollar_volume_billions",
    "first_trade_date",
    "last_trade_date",
]

top20[columns].style.format(
    {
        "avg_volume_millions": "{:,.2f}",
        "avg_dollar_volume_billions": "{:,.2f}",
        "first_trade_date": lambda value: value.strftime("%Y-%m-%d"),
        "last_trade_date": lambda value: value.strftime("%Y-%m-%d"),
    }
)

,volume_rank,permno,ticker,comnam,trading_days,avg_volume_millions,avg_dollar_volume_billions,first_trade_date,last_trade_date
0,1,70519,C,CITIGROUP INC,3017,129.66,1.34,2003-01-02,2014-12-31
1,2,59408,BAC,BANK OF AMERICA CORP,3021,106.26,1.53,2003-01-02,2014-12-31
2,3,10107,MSFT,MICROSOFT CORP,3021,60.85,1.70,2003-01-02,2014-12-31
3,4,59328,INTC,INTEL CORP,3021,58.30,1.30,2003-01-02,2014-12-31
4,5,80924,SIRI,SIRIUS X M HOLDINGS INC,3021,54.90,0.15,2003-01-02,2014-12-31
5,6,13407,FB,FACEBOOK INC,659,53.84,2.41,2012-05-18,2014-12-31
6,7,76076,CSCO,CISCO SYSTEMS INC,3021,52.55,1.10,2003-01-02,2014-12-31
7,8,83332,LU,LUCENT TECHNOLOGIES INC,987,50.43,0.15,2003-01-02,2006-11-30
8,9,12060,GE,GENERAL ELECTRIC CO,3021,47.78,1.06,2003-01-02,2014-12-31
9,10,25785,F,FORD MOTOR CO DEL,3021,45.05,0.47,2003-01-02,2014-12-31


In [5]:
plot_data = top20.sort_values("avg_volume_millions", ascending=True).copy()
plot_data["label"] = plot_data["ticker"] + " - " + plot_data["comnam"].str.title()

fig = px.bar(
    plot_data,
    x="avg_volume_millions",
    y="label",
    orientation="h",
    title="Top 20 CRSP Common Stocks By Average Daily Share Volume, 2003-2014",
    labels={
        "avg_volume_millions": "Average daily volume, millions of shares",
        "label": "",
    },
    hover_data={
        "label": False,
        "ticker": True,
        "comnam": True,
        "permno": True,
        "volume_rank": True,
        "trading_days": ":,",
        "avg_volume_millions": ":,.2f",
        "avg_dollar_volume_billions": ":,.2f",
        "first_trade_date": "|%Y-%m-%d",
        "last_trade_date": "|%Y-%m-%d",
    },
    color_discrete_sequence=["#4C78A8"],
)
fig.update_layout(
    height=700,
    hovermode="closest",
    yaxis={"categoryorder": "array", "categoryarray": plot_data["label"].tolist()},
)
fig.show()

## Top 20 Trading Volume Over Time

The bar chart above collapses the full 2003-2014 period into one average per stock. This section checks whether those top-volume stocks also look sensible through time.

It queries CRSP daily volume for the top 20 `permno` values, aggregates to monthly average daily volume, and plots each stock's monthly series in millions of shares.

In [6]:
import os

import wrds
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

WRDS_USERNAME = os.environ.get("WRDS_USERNAME")
WRDS_PASSWORD = os.environ.get("WRDS_PASSWORD")

if not WRDS_USERNAME or not WRDS_PASSWORD:
    raise ValueError("Set WRDS_USERNAME and WRDS_PASSWORD in a local .env file to run this section.")

top20_permnos = top20["permno"].astype(int).tolist()
permno_sql = ", ".join(str(permno) for permno in top20_permnos)

volume_query = f"""
select
    permno,
    date,
    vol
from crsp.dsf
where date between '{manifest['start']}' and '{manifest['end']}'
  and permno in ({permno_sql})
  and vol is not null
"""

db = wrds.Connection(wrds_username=WRDS_USERNAME, wrds_password=WRDS_PASSWORD)
try:
    top20_daily_volume = db.raw_sql(volume_query, date_cols=["date"])
finally:
    db.close()

print(f"Rows returned: {len(top20_daily_volume):,}")
top20_daily_volume.head()

Loading library list...


Done


Rows returned: 49,121


,permno,date,vol
0,10078,2003-01-02,54577229.0
1,10078,2003-01-03,54557157.0
2,10078,2003-01-06,71454157.0
3,10078,2003-01-07,57516577.0
4,10078,2003-01-08,51026646.0


In [7]:
top20_lookup = top20[["permno", "ticker", "comnam"]].copy()
volume_with_names = top20_daily_volume.merge(top20_lookup, on="permno", how="left")
volume_with_names["month"] = volume_with_names["date"].dt.to_period("M").dt.to_timestamp()
volume_with_names["volume_millions"] = volume_with_names["vol"] / 1_000_000

monthly_volume = (
    volume_with_names
    .groupby(["month", "ticker", "comnam"], as_index=False)
    .agg(
        avg_daily_volume_millions=("volume_millions", "mean"),
        trading_days=("volume_millions", "size"),
    )
)

coverage = (
    monthly_volume
    .groupby("ticker", as_index=False)
    .agg(
        months=("month", "nunique"),
        first_month=("month", "min"),
        last_month=("month", "max"),
        avg_monthly_daily_volume_millions=("avg_daily_volume_millions", "mean"),
    )
    .sort_values("avg_monthly_daily_volume_millions", ascending=False)
)

coverage

,ticker,months,first_month,last_month,avg_monthly_daily_volume_millions
2,C,144,2003-01-01,2014-12-01,130.318481
1,BAC,144,2003-01-01,2014-12-01,106.461607
12,MSFT,144,2003-01-01,2014-12-01,60.842834
8,INTC,144,2003-01-01,2014-12-01,58.324553
6,FB,32,2012-05-01,2014-12-01,55.188903
16,SIRI,144,2003-01-01,2014-12-01,54.891578
3,CSCO,144,2003-01-01,2014-12-01,52.60587
11,LU,47,2003-01-01,2006-11-01,50.555254
7,GE,144,2003-01-01,2014-12-01,47.737362
5,F,144,2003-01-01,2014-12-01,45.097957


In [8]:
monthly_plot = monthly_volume.copy()
monthly_plot["label"] = monthly_plot["ticker"] + " - " + monthly_plot["comnam"].str.title()

fig = px.line(
    monthly_plot.sort_values(["ticker", "month"]),
    x="month",
    y="avg_daily_volume_millions",
    color="ticker",
    title="Monthly Average Daily Trading Volume For Top 20 CRSP Candidates, 2003-2014",
    labels={
        "month": "Month",
        "avg_daily_volume_millions": "Average daily volume, millions of shares",
        "ticker": "Ticker",
    },
    hover_data={
        "ticker": True,
        "comnam": True,
        "month": "|%Y-%m",
        "avg_daily_volume_millions": ":,.2f",
        "trading_days": True,
    },
)
fig.update_traces(mode="lines+markers", line={"width": 1.8}, marker={"size": 4})
fig.update_layout(height=750, legend_title_text="Ticker", hovermode="closest")
fig.show()